# 06 — Interactive Evaluation Notebook

This notebook is the primary developer tool for evaluating any trained or baseline
compressor configuration against a set of travel planning requests.

---

## How to use this notebook

Run the cells **in order** from top to bottom. Each section has a small set of
**configuration variables** (ALL_CAPS) at the top of its first code cell — those
are the only things you normally need to change. Everything else is implementation.

**Typical workflow for evaluating a new checkpoint:**
1. Set `REQUEST_SPLIT` and `SELECTION_MODE` → which requests to test
2. Set `COMPRESSOR_TYPE`, `AGENT_MODE`, `CHECKPOINT_PATH` → what to evaluate
3. Set `JUDGE_MODEL_ID = None` for deterministic-only (no API key needed)
4. Run all cells → check Section 7 for aggregated scores, Section 8 for detail
5. Compare against previous runs in Section 9

---

## Key concepts

| Term | What it is |
|---|---|
| `UserRequest` | A structured travel planning request (budget, dates, hard/soft constraints) |
| `EpisodeLog` | Complete record of one agent run: trajectory, tool stats, final itinerary, reward components |
| `EvalResult` | Scores for one episode: deterministic metrics + LLM judge scores + metric version tag |
| `EvalRunManifest` | Run-level metadata: which compressor, checkpoint, seeds, and metric version were used |
| `DeterministicEvaluator` | Scores an episode purely from the `EpisodeLog` — no LLM needed |
| `LLMJudge` | Calls an LLM to score the final itinerary on rubric dimensions |
| `Evaluator` | Orchestrates both layers and returns one `EvalResult` per episode |

---

## Sections

1. **Setup** — imports, paths, metric version display
2. **Request Picker** — load and filter `UserRequest` files
3. **Model & Compressor Selector** — configure what to evaluate
4. **Episode Source** — load saved episodes or run new ones
5. **Metric Selector** — deterministic, LLM judge, rubric dimensions
6. **Run Evaluation** — score + save versioned manifest
7. **Aggregated Results** — summary table and bar chart
8. **Drill-Down** — inspect one episode in full detail
9. **Previous Runs Comparison** — cross-run table and regression detection
10. **Next Steps & Improvements** — documented opportunities for future work

---

> **Deterministic-only mode** (no API key needed): keep `JUDGE_MODEL_ID = None`.
>
> **LLM judge mode**: set `JUDGE_MODEL_ID` to any [litellm model string](https://docs.litellm.ai/docs/providers)
> and ensure the matching API key (`ANTHROPIC_API_KEY`, `OPENAI_API_KEY`, etc.) is in your environment.

---
## 1. Setup

Imports, directory paths, and metric version display.
No configuration variables here — just run the two cells.

In [ ]:
import sys
import json
import random
import uuid
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

# Make the src/ package importable when the notebook runs from notebooks/.
# This is equivalent to having run `pip install -e .` but works without it.
sys.path.insert(0, str(Path("../src").resolve()))

from optimized_llm_planning_memory.core.models import UserRequest, EvalResult
from optimized_llm_planning_memory.evaluation.evaluator import Evaluator
from optimized_llm_planning_memory.evaluation.deterministic import (
    DeterministicEvaluator,
    METRIC_VERSION,    # e.g. "v1" — stamped on every EvalResult
    METRIC_CHANGELOG,  # dict mapping version → description of what changed
)
from optimized_llm_planning_memory.evaluation.llm_judge import LLMJudge
from optimized_llm_planning_memory.evaluation.manifest import EvalRunManifest
from optimized_llm_planning_memory.utils.episode_io import (
    list_episodes,
    load_episode,
    save_eval_run,   # writes manifest.json + results.jsonl
    load_eval_run,   # reads them back
    list_eval_runs,  # returns all manifests sorted newest-first
)

# ── Directory layout ──────────────────────────────────────────────────────────
# All paths are relative to the repo root (one level up from notebooks/).
BASE_DIR     = Path("..").resolve()
REQUESTS_DIR = BASE_DIR / "data" / "user_requests"    # hand-crafted + generated requests
EPISODES_DIR = BASE_DIR / "outputs" / "episodes"       # episodes from run_episode.py
BASELINE_DIR = BASE_DIR / "outputs" / "baseline_eval" # episodes from run_baseline_eval.py
EVAL_DIR     = BASE_DIR / "outputs" / "eval_results"  # where this notebook writes results

print(f"Repo root      : {BASE_DIR}")
print(f"Requests dir   : {REQUESTS_DIR}")
print(f"Episodes dir   : {EPISODES_DIR}")
print(f"Baseline dir   : {BASELINE_DIR}")
print(f"Eval output dir: {EVAL_DIR}")

In [ ]:
# Display the active metric version and what it covers.
#
# WHY THIS MATTERS: metrics evolve over time (e.g., new constraints get added).
# Every EvalResult carries the version tag under which it was computed, so you
# can tell at a glance whether two sets of results are directly comparable.
# If you load older results with a different version tag, Section 9 will flag it.
print(f"Active metric version : {METRIC_VERSION}")
print()
print(f"Coverage:")
print(f"  {METRIC_CHANGELOG[METRIC_VERSION]}")

---
## 2. Request Picker

Load `UserRequest` JSON files from `data/user_requests/` and select a subset to evaluate.

**Configuration variables:**

| Variable | Values | Effect |
|---|---|---|
| `REQUEST_SPLIT` | `"test"` / `"train"` / `"all"` | Which subdirectory to read from |
| `SELECTION_MODE` | `"manual"` | Select specific indices from the printed list |
| | `"random"` | Sample `RANDOM_N` requests at random |
| `MANUAL_INDICES` | list of ints | e.g. `[0, 2, 5]` — only used when `SELECTION_MODE="manual"` |
| `RANDOM_N` | int | Number to sample — only used when `SELECTION_MODE="random"` |

Run **Cell 3** first to see the numbered list of available requests, then set your
indices in **Cell 4** accordingly.

In [ ]:
def load_requests(requests_dir: Path, split: str = "test") -> list[UserRequest]:
    """
    Load all UserRequest JSON files from data/user_requests/{split}/.

    Files that fail to parse are skipped with a warning rather than crashing,
    so a single malformed file does not block the whole evaluation.
    """
    if split == "all":
        # Recursively find all JSON files regardless of train/test split
        globs = list(requests_dir.glob("**/*.json"))
    else:
        globs = list((requests_dir / split).glob("*.json"))

    results = []
    for p in sorted(globs):
        try:
            data = json.loads(p.read_text(encoding="utf-8"))
            results.append(UserRequest.model_validate(data))
        except Exception as e:
            print(f"  [skip] {p.name}: {e}")
    return results


# ── CONFIGURE ─────────────────────────────────────────────────────────────────
REQUEST_SPLIT  = "test"   # "train" | "test" | "all"
SELECTION_MODE = "manual" # "manual" | "random"
MANUAL_INDICES = [0]      # used when SELECTION_MODE=="manual"; change after seeing the list below
RANDOM_N       = 3        # used when SELECTION_MODE=="random"
# ─────────────────────────────────────────────────────────────────────────────

all_requests = load_requests(REQUESTS_DIR, split=REQUEST_SPLIT)

print(f"Found {len(all_requests)} request(s) in split='{REQUEST_SPLIT}'")
print()
for i, r in enumerate(all_requests):
    n_hard = len(r.hard_constraints)
    n_soft = len(r.soft_constraints)
    print(
        f"  [{i}] {r.request_id:<38}  "
        f"dest={r.destination_cities}  "
        f"${r.budget_usd:.0f}  "
        f"{r.start_date} to {r.end_date}  "
        f"hard={n_hard} soft={n_soft}"
    )
print()
print("Set MANUAL_INDICES to the index numbers you want, then run the next cell.")

In [ ]:
def pick_requests(
    all_reqs: list[UserRequest],
    mode: str,
    manual_indices: list[int],
    random_n: int,
) -> list[UserRequest]:
    """Select a subset of requests by index or random sampling."""
    if mode == "manual":
        return [all_reqs[i] for i in manual_indices if i < len(all_reqs)]
    elif mode == "random":
        return random.sample(all_reqs, min(random_n, len(all_reqs)))
    else:
        raise ValueError(f"Unknown selection mode: {mode!r}. Use 'manual' or 'random'.")


selected_requests = pick_requests(all_requests, SELECTION_MODE, MANUAL_INDICES, RANDOM_N)

print(f"Selected {len(selected_requests)} request(s):")
for r in selected_requests:
    print(f"  {r.request_id}")
    print(f"    {r.raw_text[:120]}...")

In [ ]:
def load_matching_episodes(episodes_dir: Path, baseline_dir: Path, requests: list[UserRequest]):
    """
    Find pre-existing EpisodeLog files whose request_id matches the selected requests.

    Searches both the standard episodes/ directory and baseline_eval/ so you can
    evaluate episodes produced by either run_episode.py or run_baseline_eval.py.
    If the same episode_id appears in both directories, the last-loaded version wins
    (deduplication by episode_id).
    """
    target_ids = {r.request_id for r in requests}
    all_eps = []

    for search_dir in [episodes_dir, baseline_dir]:
        if not search_dir.exists():
            continue
        # Both directories use ep_*.json and episode_*.json naming conventions
        for pattern in ["ep_*.json", "episode_*.json"]:
            for p in sorted(search_dir.glob(pattern)):
                try:
                    ep = load_episode(p)
                    if ep.request_id in target_ids:
                        all_eps.append(ep)
                except Exception:
                    pass  # Skip unreadable files silently

    # Deduplicate: keep the last occurrence of each episode_id
    seen = {}
    for ep in all_eps:
        seen[ep.episode_id] = ep
    return list(seen.values())


selected_episodes = load_matching_episodes(EPISODES_DIR, BASELINE_DIR, selected_requests)

print(f"Found {len(selected_episodes)} matching episode(s) on disk:")
for ep in selected_episodes:
    itinerary_status = "itinerary OK" if ep.final_itinerary else "no itinerary"
    print(
        f"  {ep.episode_id}  "
        f"request={ep.request_id}  "
        f"success={ep.success}  "
        f"steps={ep.total_steps}  "
        f"{itinerary_status}"
    )

if not selected_episodes:
    print("No episodes found on disk. Section 4 will run new ones.")

---
## 3. Model & Compressor Selector

Configure which compressor and agent mode you are evaluating.

**Compressor types:**
- `"identity"` — full trajectory passed through unchanged (RAW baseline, no compression)
- `"llm"` — a frozen LLM summarises the trajectory at each step (LLM Summary baseline)
- `"transformer"` — the PPO-trained seq2seq compressor (requires a checkpoint)
- `"dummy"` — small randomly-initialised seq2seq; useful for pipeline smoke tests

**Agent modes:**
- `"raw"` — agent receives the full trajectory (pairs with `identity` compressor)
- `"llm_summary"` — agent receives the LLM summary (pairs with `llm` compressor)
- `"compressor"` — agent receives the trained compressed state (pairs with `transformer`)

**LLM judge config:**  
Leave `JUDGE_MODEL_ID = None` to skip the judge entirely (deterministic metrics only, no API cost).  
Set it to any [litellm model string](https://docs.litellm.ai/docs/providers) to enable rubric scoring.

In [ ]:
# ── CONFIGURE ─────────────────────────────────────────────────────────────────
COMPRESSOR_TYPE = "identity"           # "identity" | "llm" | "transformer" | "dummy"
AGENT_MODE      = "raw"                # "raw" | "llm_summary" | "compressor"
CHECKPOINT_PATH = None                 # Path to a .zip SB3 checkpoint, or None
                                       # e.g. "../outputs/checkpoints/final/ppo_model.zip"
LLM_MODEL_ID    = "claude-sonnet-4-6" # Agent LLM — only used when running new episodes
WORLD_SEED      = 42                   # Simulator seed for new episodes

# LLM judge — set to None to skip (deterministic-only mode)
JUDGE_MODEL_ID  = None                 # e.g. "anthropic/claude-sonnet-4-6"
# ─────────────────────────────────────────────────────────────────────────────

print(f"Compressor     : {COMPRESSOR_TYPE}")
print(f"Agent mode     : {AGENT_MODE}")
print(f"Checkpoint     : {CHECKPOINT_PATH or 'none (using default init)'}")
print(f"Agent LLM      : {LLM_MODEL_ID}")
print(f"World seed     : {WORLD_SEED}")
print()
if JUDGE_MODEL_ID:
    print(f"LLM judge      : ENABLED ({JUDGE_MODEL_ID})")
    print("  Make sure the corresponding API key is set in your environment.")
else:
    print("LLM judge      : DISABLED (deterministic metrics only — no API key needed)")

---
## 4. Episode Source

If Section 2 found existing `EpisodeLog` files for the selected requests, they are used
directly — no agent is run.

If nothing was found, this section runs the **scripted baseline agent**
(`scripts/run_baseline_eval.py`) to generate new episodes. The scripted agent does not
call an LLM — it follows a fixed tool-call sequence, making it useful for smoke tests
and debugging the evaluation pipeline without API costs.

> **To evaluate a real LLM agent**: run `python scripts/run_episode.py` from the terminal
> with your desired config overrides, then re-run Section 2 — the new episodes will be
> picked up automatically.

At the end of this section, `ep_req_pairs` contains the aligned
`(EpisodeLog, UserRequest)` tuples that Section 6 will score.

In [ ]:
def run_scripted_episodes(requests, world_seed):
    """
    Run the deterministic scripted agent for each request and save the episodes.

    The scripted agent is imported from scripts/run_baseline_eval.py at runtime
    rather than being a package import, because it is a standalone script.
    It calls: get_available_routes → search_hotels → book_hotel →
               search_attractions → search_restaurants → search_events → book_event

    Produced EpisodeLogs are saved to outputs/episodes/ and returned.
    """
    import importlib.util
    from optimized_llm_planning_memory.simulator.adapter import SimulatorAdapter
    from optimized_llm_planning_memory.utils.episode_io import save_episode

    # Dynamically load the script so we can call run_scripted_episode()
    spec = importlib.util.spec_from_file_location(
        "run_baseline_eval",
        str(BASE_DIR / "scripts" / "run_baseline_eval.py"),
    )
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)

    episodes = []
    for req in requests:
        print(f"  Running scripted episode for {req.request_id}...")
        try:
            # A fresh SimulatorAdapter per episode ensures clean state.
            # The seed controls which hotels/events/attractions are generated.
            sim = SimulatorAdapter(seed=world_seed)
            ep = mod.run_scripted_episode(sim, req)
            save_episode(ep, EPISODES_DIR)
            episodes.append(ep)
            print(f"    success={ep.success}  steps={ep.total_steps}")
        except Exception as e:
            print(f"    ERROR: {e}")
    return episodes


# Run new episodes only if none were found on disk
if not selected_episodes:
    print("No pre-existing episodes found. Running scripted baseline agent...")
    print()
    selected_episodes = run_scripted_episodes(selected_requests, WORLD_SEED)
    print()
    print(f"Ran {len(selected_episodes)} new episode(s).")
else:
    print(f"Using {len(selected_episodes)} episode(s) loaded from disk.")

# Align episodes with their corresponding UserRequests by request_id.
# An episode without a matching request is silently dropped — this can happen
# if episodes were generated for a different request split.
req_map = {r.request_id: r for r in selected_requests}
ep_req_pairs = [
    (ep, req_map[ep.request_id])
    for ep in selected_episodes
    if ep.request_id in req_map
]

print()
print(f"Ready to evaluate {len(ep_req_pairs)} episode-request pair(s).")
if len(ep_req_pairs) < len(selected_requests):
    missing = set(r.request_id for r in selected_requests) - {ep.request_id for ep, _ in ep_req_pairs}
    print(f"WARNING: no episodes found for: {missing}")

---
## 5. Metric Selector

Choose which evaluation layers to run.

**Deterministic metrics** (always recommended):

| Metric | What it measures |
|---|---|
| `hard_constraint_ratio` | Fraction of budget / date / hotel-star constraints satisfied |
| `soft_constraint_score` | Weighted score for preferences (dining, activities, events) |
| `budget_adherence` | 1.0 if within budget; penalised proportionally if over |
| `logical_consistency` | Date ordering correct + no duplicate hotel bookings |
| `tool_efficiency` | 1 − (redundant calls / total calls) |
| `tool_failure_rate` | Fraction of tool calls that errored |
| `avg_tool_latency_ms` | Mean time per tool call |
| `steps_per_episode` | Total ReAct steps |

**LLM judge** (optional, requires API key):

Scores the final itinerary on six qualitative rubric dimensions. Each dimension returns
a score in [0, 1] **plus a free-text reasoning** stored in `EvalResult.rubric_breakdown`.
This is what you drill into in Section 8 when metrics look surprising.

Set `DEVELOPER_NOTES` to anything useful — it's stored in the manifest and helps you
remember what was different about this run when you look at it six weeks later.

In [ ]:
# ── CONFIGURE ─────────────────────────────────────────────────────────────────
USE_DETERMINISTIC = True
USE_LLM_JUDGE = JUDGE_MODEL_ID is not None  # auto-derived from Section 3 config

# Rubric dimensions for the LLM judge.
# All six are included by default. Remove any you don't care about to reduce
# API cost and latency — each dimension is a separate structured LLM call.
RUBRIC_DIMENSIONS = [
    "constraint_satisfaction",  # did the plan honour hard and soft constraints?
    "itinerary_feasibility",    # are timings, locations, and logistics realistic?
    "preference_alignment",     # does the plan reflect the traveller's stated preferences?
    "diversity_balance",        # is there a good mix of activity types?
    "budget_efficiency",        # does the plan make good use of the available budget?
    "overall_quality",          # holistic judgement of the itinerary as a travel plan
]

# Optional free-text note stored in the run manifest (visible in Section 9)
DEVELOPER_NOTES = ""  # e.g. "first run after fixing constraint engine preference stub"
# ─────────────────────────────────────────────────────────────────────────────

print(f"Deterministic metrics : {'ON' if USE_DETERMINISTIC else 'OFF'}  (metric version: {METRIC_VERSION})")
print(f"LLM judge             : {'ON' if USE_LLM_JUDGE else 'OFF'}" +
      (f"  (model: {JUDGE_MODEL_ID})" if USE_LLM_JUDGE else ""))
if USE_LLM_JUDGE:
    print(f"Rubric dimensions     : {RUBRIC_DIMENSIONS}")
    print(f"  Estimated API calls : {len(ep_req_pairs)} episodes x {len(RUBRIC_DIMENSIONS)} dimensions")
if DEVELOPER_NOTES:
    print(f"Notes                 : {DEVELOPER_NOTES}")

---
## 6. Run Evaluation

Scores each `(EpisodeLog, UserRequest)` pair and saves the run to disk.

**What this section produces:**
- `eval_results` — a list of `EvalResult` objects (one per episode)
- `run_id` — a short unique ID for this run
- `outputs/eval_results/{run_id}/manifest.json` — run-level metadata
- `outputs/eval_results/{run_id}/results.jsonl` — per-episode scores

Saving to disk lets Section 9 compare this run against previous ones, and lets
you reload and re-analyse results later without re-running any agents.

In [ ]:
# Build the evaluator from the selected layers.
# Evaluator.evaluate_episode() calls both layers in sequence and returns one EvalResult.
det_eval = DeterministicEvaluator() if USE_DETERMINISTIC else None
judge    = LLMJudge(JUDGE_MODEL_ID, rubric_dimensions=RUBRIC_DIMENSIONS) if USE_LLM_JUDGE else None
evaluator = Evaluator(deterministic_eval=det_eval, llm_judge=judge)

eval_results: list[EvalResult] = []

for ep, req in tqdm(ep_req_pairs, desc="Evaluating episodes"):
    # evaluate_episode() is stateless — safe to parallelise with ThreadPoolExecutor
    # if you have many episodes and the LLM judge is the bottleneck.
    result = evaluator.evaluate_episode(ep, req)
    eval_results.append(result)

print(f"Done. Evaluated {len(eval_results)} episode(s).")

In [ ]:
# Construct the run manifest and save everything to disk.
#
# The manifest records WHO produced WHAT with WHICH config — without it,
# you would not be able to tell two runs apart six weeks later.
# `config_hash` is set to "manual" here because we're outside Hydra;
# scripts/run_evaluation.py fills in an actual MD5 of the Hydra config.

run_id = str(uuid.uuid4())[:8]  # short ID, e.g. "3fa8c10b"

manifest = EvalRunManifest(
    run_id=run_id,
    created_at=datetime.now(timezone.utc).isoformat(),
    compressor_type=COMPRESSOR_TYPE,
    agent_mode=AGENT_MODE,
    judge_model_id=str(JUDGE_MODEL_ID or "none"),
    checkpoint_path=str(CHECKPOINT_PATH) if CHECKPOINT_PATH else None,
    config_hash="manual",
    metric_version=METRIC_VERSION,
    request_ids=[r.request_id for r in selected_requests],
    n_episodes=len(eval_results),
    deterministic_only=not USE_LLM_JUDGE,
    world_seeds=[WORLD_SEED],
    notes=DEVELOPER_NOTES or None,
)

run_dir = save_eval_run(manifest, eval_results, EVAL_DIR)

print(f"Run ID  : {run_id}")
print(f"Saved to: {run_dir}")
print(f"  manifest.json  — metadata")
print(f"  results.jsonl  — {len(eval_results)} EvalResult record(s)")

---
## 7. Aggregated Results

Summarise scores across all evaluated episodes.

**Reading the table:**
- `mean` — average across episodes; the headline number for comparing configurations
- `std` — standard deviation; high std means results vary a lot across requests
  (either the requests differ in difficulty, or the agent is inconsistent)

**Reading the chart:**
- All metrics are normalised to [0, 1] (higher is always better)
- `hard_constraint_ratio` and `budget_adherence` are the non-negotiable success criteria
- `soft_constraint_score` is often low (~0.17) because the constraint engine's
  preference matcher is stubbed — see Section 10.1 for the fix
- `overall_score` is a weighted average that double-counts `hard_constraint_ratio`

In [ ]:
# Compute mean ± std for every metric across all episodes.
# Evaluator.aggregate() returns a flat dict with keys like
# "hard_constraint_ratio_mean", "hard_constraint_ratio_std", etc.
agg = evaluator.aggregate(eval_results)

mean_rows = {k: v for k, v in agg.items() if k.endswith("_mean")}
std_rows  = {k: v for k, v in agg.items() if k.endswith("_std")}

df_agg = pd.DataFrame({
    "metric": [k.replace("_mean", "") for k in mean_rows],
    "mean":   list(mean_rows.values()),
    "std":    [std_rows.get(k.replace("_mean", "_std"), 0.0) for k in mean_rows],
})
df_agg = df_agg.set_index("metric").sort_index()

print(f"Aggregated over {len(eval_results)} episode(s)  "
      f"[{COMPRESSOR_TYPE} / {AGENT_MODE} / metric_v={METRIC_VERSION}]")
print()
display(df_agg.style.format("{:.4f}").bar(subset=["mean"], color="steelblue"))

In [ ]:
# Per-episode table — useful for spotting outliers and understanding variance.
# LLM judge scores are prefixed with "judge_" to distinguish them from deterministic ones.
rows = []
for r in eval_results:
    row = {
        "episode_id":    r.episode_id,
        "request_id":    r.request_id,
        "overall_score": r.overall_score,
        "metric_ver":    r.metric_version,  # tag for cross-version comparison safety
        **r.deterministic_scores,
        **{f"judge_{k}": v for k, v in r.llm_judge_scores.items()},
    }
    rows.append(row)

df_episodes = pd.DataFrame(rows)
display(df_episodes)

In [ ]:
# Bar chart of mean scores for key metrics.
# Skip any metric that isn't in df_episodes (e.g. judge_ metrics when judge is disabled).
key_metrics = [
    "hard_constraint_ratio",
    "soft_constraint_score",
    "budget_adherence",
    "logical_consistency",
    "tool_efficiency",
    "overall_score",
]

plot_vals = [(m, df_episodes[m].mean()) for m in key_metrics if m in df_episodes.columns]

if plot_vals:
    labels, vals = zip(*plot_vals)
    fig, ax = plt.subplots(figsize=(10, 4))
    bars = ax.bar(labels, vals, color="steelblue", alpha=0.85, edgecolor="white")
    ax.set_ylim(0, 1.1)
    ax.set_ylabel("Score (0–1, higher is better)")
    ax.set_title(
        f"Mean Scores — compressor={COMPRESSOR_TYPE} / mode={AGENT_MODE} "
        f"(n={len(eval_results)}, metric_v={METRIC_VERSION})"
    )
    ax.axhline(0.5, color="grey", linestyle="--", linewidth=0.8, alpha=0.6)  # 50% reference line
    ax.tick_params(axis="x", rotation=30)
    for bar, val in zip(bars, vals):
        ax.text(
            bar.get_x() + bar.get_width() / 2, val + 0.02,
            f"{val:.3f}", ha="center", va="bottom", fontsize=9,
        )
    plt.tight_layout()
    plt.show()
else:
    print("No metrics to plot — check that eval_results is non-empty.")

---
## 8. Drill-Down

Inspect a single episode in full detail to understand *why* it scored the way it did.

**How to use this section:**
1. Look at the per-episode table in Section 7 and identify an interesting episode
   (e.g., the one with the lowest `hard_constraint_ratio`)
2. Set `EPISODE_INDEX` to that episode's row index
3. Run all four cells in this section

**What you will see:**
- Episode header: ID, request, success flag, step count, overall score
- Full ReAct trajectory: every thought, tool call, and observation
- ASCII bar chart of all deterministic scores
- Rubric breakdown with the judge's reasoning (if LLM judge was enabled)

In [ ]:
# ── CONFIGURE ─────────────────────────────────────────────────────────────────
EPISODE_INDEX = 0  # Row index into ep_req_pairs / eval_results (0 = first episode)
# ─────────────────────────────────────────────────────────────────────────────

if not ep_req_pairs or not eval_results:
    print("No episodes to display. Run Sections 2–6 first.")
else:
    ep_detail, req_detail = ep_req_pairs[EPISODE_INDEX]
    result_detail = eval_results[EPISODE_INDEX]

    print("=" * 60)
    print(f"Episode   : {ep_detail.episode_id}")
    print(f"Request   : {req_detail.request_id}")
    print(f"Agent mode: {ep_detail.agent_mode}")
    print(f"Success   : {ep_detail.success}  |  Steps: {ep_detail.total_steps}")
    print(f"Overall   : {result_detail.overall_score:.4f}  |  Metric v{result_detail.metric_version}")
    print("=" * 60)
    print()
    print(f"Request text:")
    print(f"  {req_detail.raw_text}")
    print()
    print(f"Hard constraints:")
    for c in req_detail.hard_constraints:
        print(f"  [{c.constraint_id}] {c.description}")
    print()
    print(f"Soft constraints:")
    for c in req_detail.soft_constraints:
        print(f"  [{c.constraint_id}] {c.description}")

In [ ]:
# Print the full ReAct trajectory: every thought, tool call, and observation.
# This is the most detailed view available — use it to understand exactly what
# the agent did and why it succeeded or failed on specific constraints.
if ep_req_pairs and eval_results:
    try:
        from optimized_llm_planning_memory.utils.visualization import episode_to_string
        print(episode_to_string(ep_detail))
    except Exception as e:
        # Fall back to the raw trajectory text if visualization is unavailable
        print(f"[visualization unavailable: {e}]")
        print(ep_detail.trajectory.to_text())

In [ ]:
# Per-metric ASCII bar chart — quick visual scan of where the episode scored well/poorly.
# Each # represents 1/30 of a point, so 30 hashes = perfect score.
if eval_results:
    print("Deterministic scores:")
    print("-" * 60)
    for metric, value in sorted(result_detail.deterministic_scores.items()):
        # Metrics like avg_tool_latency_ms and steps_per_episode are unbounded —
        # clamp bar to [0, 30] so they don't produce misleading long bars.
        bar_len = min(int(value * 30), 30) if value <= 1.0 else min(int(value / 10), 30)
        bar = "#" * bar_len
        print(f"  {metric:<30} {value:>8.3f}  {bar}")

    if result_detail.llm_judge_scores:
        print()
        print("LLM judge scores:")
        print("-" * 60)
        for metric, value in sorted(result_detail.llm_judge_scores.items()):
            bar = "#" * int(value * 30)
            print(f"  {metric:<30} {value:>8.3f}  {bar}")

In [ ]:
# Rubric breakdown — the judge's reasoning for each dimension.
# This is the most actionable output when a score surprises you:
# read the reasoning to understand what the judge penalised.
#
# EvalResult.rubric_breakdown is only populated when USE_LLM_JUDGE=True.
# It maps dimension_name → {"score": float, "reasoning": str}.
if eval_results:
    if result_detail.rubric_breakdown:
        print("Rubric breakdown (LLM judge reasoning):")
        print("=" * 60)
        for dim, info in sorted(result_detail.rubric_breakdown.items()):
            print(f"\n[{dim}]  score = {info['score']:.2f}")
            print(f"  {info['reasoning']}")
    elif USE_LLM_JUDGE:
        print("No rubric breakdown recorded — the judge may have returned an empty response.")
    else:
        print("LLM judge was disabled.")
        print("Set JUDGE_MODEL_ID in Section 3 and re-run Sections 6–8 to see rubric reasoning.")

---
## 9. Previous Runs Comparison

List all previously saved eval runs from `outputs/eval_results/` and build a
side-by-side comparison table.

**Why this matters for ablation studies:**  
Each row in the comparison table is one `(compressor, agent_mode, checkpoint)` configuration.
To compare RAW vs COMPRESSOR conditions, run this notebook twice with different
Section 3 settings — the comparison table will show both runs together.

**Regression detection:**  
The last cell warns if the current run's `overall_score` dropped more than 5 percentage
points below the most recent run with the same `compressor_type + agent_mode`. This catches
regressions introduced by code changes without requiring a dedicated CI step.

**Metric version mismatch:**  
The comparison table shows the `metric_version` of each run. If versions differ, scores
may not be directly comparable — the table will show which runs are potentially mismatched.

In [ ]:
# List all saved manifests, newest first.
all_manifests = list_eval_runs(EVAL_DIR)

print(f"Found {len(all_manifests)} eval run(s) in {EVAL_DIR}:")
print()
for m in all_manifests:
    tag = "  <-- current run" if m.run_id == run_id else ""
    ver_warn = "  [METRIC VERSION DIFFERS]" if m.metric_version != METRIC_VERSION else ""
    print(
        f"  [{m.run_id}]  {m.created_at[:19]}  "
        f"{m.compressor_type}/{m.agent_mode}  "
        f"v={m.metric_version}  "
        f"n={m.n_episodes}  "
        f"det_only={m.deterministic_only}"
        f"{ver_warn}{tag}"
    )

In [ ]:
def build_comparison_df(eval_dir: Path, manifests: list) -> pd.DataFrame:
    """
    Build a one-row-per-run comparison DataFrame.

    Loads each run's results.jsonl and computes per-run mean scores.
    The `metric_version` column shows whether scores are directly comparable.
    """
    rows = []
    for m in manifests:
        try:
            _, results = load_eval_run(m.run_id, eval_dir)
        except Exception:
            continue
        if not results:
            continue

        def mean_of(key):
            vals = [r.deterministic_scores.get(key, 0.0) for r in results]
            return sum(vals) / len(vals)

        metric_vers = ", ".join(sorted({r.metric_version for r in results}))
        rows.append({
            "run_id":          m.run_id,
            "timestamp":       m.created_at[:19],
            "compressor":      m.compressor_type,
            "agent_mode":      m.agent_mode,
            "metric_version":  metric_vers,
            "n_episodes":      m.n_episodes,
            "overall_mean":    sum(r.overall_score for r in results) / len(results),
            "hard_con_mean":   mean_of("hard_constraint_ratio"),
            "soft_con_mean":   mean_of("soft_constraint_score"),
            "budget_mean":     mean_of("budget_adherence"),
            "efficiency_mean": mean_of("tool_efficiency"),
            "det_only":        m.deterministic_only,
            "notes":           m.notes or "",
        })

    if not rows:
        return pd.DataFrame()
    return pd.DataFrame(rows).sort_values("overall_mean", ascending=False).reset_index(drop=True)


# Compare the 5 most recent runs.
# To compare specific runs, filter all_manifests by run_id before passing here.
compare_manifests = all_manifests[:5]

if compare_manifests:
    df_compare = build_comparison_df(EVAL_DIR, compare_manifests)
    if df_compare.empty:
        print("No results loaded — check that results.jsonl files exist in the run directories.")
    else:
        float_cols = ["overall_mean", "hard_con_mean", "soft_con_mean",
                      "budget_mean", "efficiency_mean"]
        display(
            df_compare.style
            .format({c: "{:.4f}" for c in float_cols})
            .highlight_max(subset=float_cols, color="#c6efce")  # green = best
            .highlight_min(subset=float_cols, color="#ffc7ce")  # red = worst
        )
else:
    print("No previous runs to compare.")

In [ ]:
# Regression detection: compare the current run against the most recent previous run
# with the same compressor_type and agent_mode.
#
# Threshold of -0.05 means: warn if overall_score dropped by more than 5 percentage
# points. Adjust REGRESSION_THRESHOLD to taste.
REGRESSION_THRESHOLD = -0.05

same_config_runs = [
    m for m in all_manifests
    if m.compressor_type == COMPRESSOR_TYPE
    and m.agent_mode == AGENT_MODE
    and m.run_id != run_id
]

if not same_config_runs:
    print(f"No previous runs for {COMPRESSOR_TYPE}/{AGENT_MODE} to compare against.")
    print("Run this notebook again after making changes to see regression detection.")
elif not eval_results:
    print("No eval results in current run.")
else:
    # Load the most recent previous run (all_manifests is sorted newest-first)
    prev_manifest = same_config_runs[0]

    if prev_manifest.metric_version != METRIC_VERSION:
        print(
            f"WARNING: previous run [{prev_manifest.run_id}] used metric_v={prev_manifest.metric_version} "
            f"but current run uses metric_v={METRIC_VERSION}. "
            f"Regression comparison may not be meaningful."
        )

    _, prev_results = load_eval_run(prev_manifest.run_id, EVAL_DIR)

    if prev_results:
        prev_mean = sum(r.overall_score for r in prev_results) / len(prev_results)
        curr_mean = sum(r.overall_score for r in eval_results) / len(eval_results)
        delta = curr_mean - prev_mean
        direction = f"{delta:+.4f}"

        if delta < REGRESSION_THRESHOLD:
            print(
                f"REGRESSION DETECTED: overall_score {direction} vs run [{prev_manifest.run_id}] "
                f"({prev_mean:.4f} -> {curr_mean:.4f})."
            )
            print("  Check for: changed constraint weights, broken tool calls, shorter max_steps.")
        else:
            print(
                f"No regression vs previous run [{prev_manifest.run_id}] "
                f"(delta={direction}, {prev_mean:.4f} -> {curr_mean:.4f})."
            )

---
## 10. Next Steps & Improvements

These are documented improvement opportunities for future evaluation iterations.
They are recorded here rather than in a separate doc so they stay visible
alongside the code that implements the current version.

---

### 10.1 Fix `ConstraintSatisfactionEngine` preference stubs

**Current behaviour:** `_evaluate_preference()` in `core/constraints.py` always returns
`satisfied=False, score=0.5` regardless of the itinerary content. This is why
`soft_constraint_score` is stuck at ~0.17 in baseline runs.

**Fix:** Implement actual matching — check that:
- Booked `ActivityBooking.category` values overlap with `UserRequest.preferences` keywords
- `RestaurantOption.cuisine_types` match `sc-dining` soft constraint value
- `EventOption.base_ticket_price` is below the `sc-events` max price threshold

Similarly, `_evaluate_group()` returns `satisfied=True` without checking `traveler_profile.num_children`
against venue age restrictions.

---

### 10.2 Multi-seed averaging

The travel world is seeded, meaning a single seed always generates the same hotels,
events, and prices. A model that happens to perform well on seed 42 may not generalise.

**Fix:** Run each configuration N times (e.g., 5 seeds), aggregate across seeds, and
report `mean ± std`. The `EvalRunManifest.world_seeds` field is already a list for this purpose.

```python
# Example: run 5 seeds and average
all_results = []
for seed in [42, 123, 456, 789, 1337]:
    episodes = run_scripted_episodes(selected_requests, world_seed=seed)
    results = [evaluator.evaluate_episode(ep, req) for ep, req in zip(episodes, selected_requests)]
    all_results.extend(results)
agg_multi_seed = evaluator.aggregate(all_results)
```

---

### 10.3 Statistical significance testing

For comparing RAW vs COMPRESSOR conditions, a simple mean comparison is not sufficient
for a paper claim. Use the **paired Wilcoxon signed-rank test** (non-parametric, suitable
for small samples, does not assume normality):

```python
from scipy.stats import wilcoxon
raw_scores  = [r.overall_score for r in raw_results]
comp_scores = [r.overall_score for r in compressor_results]
stat, p_value = wilcoxon(raw_scores, comp_scores)
print(f"p-value = {p_value:.4f}  ({'significant' if p_value < 0.05 else 'not significant'} at alpha=0.05)")
```

---

### 10.4 Token efficiency metrics (metric v2)

`EpisodeLog.total_tokens_used` is already recorded but not evaluated. Adding these to
`DeterministicEvaluator` as a `v2` metric set would directly measure the compression benefit:

| Proposed metric | Formula |
|---|---|
| `tokens_per_episode` | `total_tokens_used` |
| `tokens_per_hard_constraint_satisfied` | `total_tokens_used / (n_satisfied + 1)` |
| `compression_ratio` | `compressed_tokens / raw_tokens` (only meaningful for compressor mode) |

When adding v2 metrics, increment `METRIC_VERSION` and add an entry to `METRIC_CHANGELOG`
in `evaluation/deterministic.py` — the version tag on old results will flag that they
are missing these metrics.

---

### 10.5 Per-difficulty-bucket analysis

Not all requests are equally hard. A useful diagnostic is to segment `df_episodes` by
request difficulty:

```python
def difficulty(req: UserRequest) -> str:
    n_hard = len(req.hard_constraints)
    budget_tightness = req.budget_usd / 400  # rough cost-per-day estimate
    if n_hard >= 4 or budget_tightness < 1.5:
        return "hard"
    elif n_hard >= 3:
        return "medium"
    return "easy"

df_episodes["difficulty"] = [difficulty(req) for _, req in ep_req_pairs]
df_episodes.groupby("difficulty")[["hard_constraint_ratio", "overall_score"]].mean()
```

---

### 10.6 Metric version migration

When loading older results in Section 9, `build_comparison_df()` already shows the
`metric_version` column. A future improvement is to add a `recompute_scores()` helper
that re-runs `DeterministicEvaluator` on saved `EpisodeLog` files using the current
metric version — upgrading old results without re-running any agents:

```python
def recompute_scores(run_id, eval_dir, episodes_dir):
    manifest, old_results = load_eval_run(run_id, eval_dir)
    # reload EpisodeLogs and re-score with current DeterministicEvaluator
    ...
```

---

### 10.7 Ablation runner integration

For systematic ablation studies, use `AblationRunner` from `evaluation/ablation.py`
instead of running this notebook manually for each configuration:

```python
from optimized_llm_planning_memory.evaluation.ablation import AblationRunner

def generate(overrides: dict):
    """Run episodes for each config override and return (episodes, requests)."""
    episodes = run_scripted_episodes(selected_requests, world_seed=WORLD_SEED)
    return episodes, selected_requests

runner = AblationRunner(evaluator=evaluator, episode_generator=generate)
ablation_results = runner.run(axes={
    "compressor_type": ["identity", "llm"],
    "agent_mode":      ["raw", "llm_summary"],
})
AblationRunner.print_summary(ablation_results, metric="overall_score_mean")
```

Each `AblationResult` also has an `aggregated_scores` dict you can feed directly into
`build_comparison_df()` for the comparison table.